In [ ]:
import hashlib

# Standard Base58 alphabet (excludes 0, O, I, l, +, /)
BASE58_ALPHABET = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"


def _double_sha256(data: bytes) -> bytes:
    """Helper: Computes SHA256(SHA256(data))"""
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()


def b58encode(raw_bytes: bytes) -> str:
    """Encodes raw bytes into a standard Base58 string."""
    # Count leading zero bytes to preserve them as '1' characters
    leading_zeros = 0
    for byte in raw_bytes:
        if byte == 0:
            leading_zeros += 1
        else:
            break

    # Convert bytes to an integer (big-endian)
    num = int.from_bytes(raw_bytes, byteorder="big")

    # Base conversion math
    result = []
    while num > 0:
        num, remainder = divmod(num, 58)
        result.append(BASE58_ALPHABET[remainder])

    encoded = "".join(reversed(result))

    # Prepend '1' for each leading zero byte
    return ("1" * leading_zeros) + encoded


def b58decode(base58_str: str) -> bytes:
    """Decodes a Base58 string back into raw bytes."""
    # Count leading '1's
    leading_ones = 0
    for char in base58_str:
        if char == "1":
            leading_ones += 1
        else:
            break

    # Convert Base58 string back to an integer
    num = 0
    for char in base58_str:
        if char not in BASE58_ALPHABET:
            raise ValueError(f"Invalid character '{char}' in Base58 string")
        num = num * 58 + BASE58_ALPHABET.index(char)

    # Convert integer back to bytes
    num_bytes = num.to_bytes((num.bit_length() + 7) // 8, byteorder="big") if num > 0 else b""

    # Prepend null bytes for leading '1's
    return (b"\x00" * leading_ones) + num_bytes


def b58check_encode(version: bytes, payload: bytes) -> str:
    """Encodes version + payload into a Base58Check string with a 4-byte checksum."""
    data = version + payload
    
    # 1. Double SHA-256 and extract first 4 bytes as checksum
    checksum = _double_sha256(data)[:4]
    
    # 2. Append checksum and encode
    return b58encode(data + checksum)


def b58check_decode(base58_str: str) -> tuple[bytes, bytes]:
    """Decodes and validates a Base58Check string.
    
    Returns: (version, payload)
    Raises: ValueError if checksum validation fails.
    """
    raw_data = b58decode(base58_str)

    if len(raw_data) < 5:  # Version (1B) + Checksum (4B) = min 5 bytes
        raise ValueError("Decoded data is too short to contain a valid checksum")

    # Split into components: Data + Checksum
    data = raw_data[:-4]
    provided_checksum = raw_data[-4:]

    # Verify Checksum
    expected_checksum = _double_sha256(data)[:4]
    if provided_checksum != expected_checksum:
        raise ValueError("Invalid Base58Check checksum! Data is corrupt or mistyped.")

    version = data[:1]
    payload = data[1:]
    return version, payload

In [2]:
if __name__ == "__main__":
    # Example 1: Creating a Bitcoin Mainnet Address
    version_byte = b"\x00"  # 0x00 = Bitcoin Mainnet P2PKH
    # Mock 20-byte HASH160 payload (Public Key Hash)
    mock_hash160 = bytes.fromhex("f54a5851e9372b87810a8e60cdd2e7cfd80b6e31")

    # Encode
    address = b58check_encode(version_byte, mock_hash160)
    print(f"Encoded Address: {address}")
    # Output: Starts with '1' because version byte is 0x00

    # Example 2: Decoding & Validating
    try:
        ver, payload = b58check_decode(address)
        print(f"Validation Result: Success!")
        print(f"Version Byte: 0x{ver.hex()}")
        print(f"Payload: {payload.hex()}")
    except ValueError as e:
        print(f"Validation Failed: {e}")

    # Example 3: Simulating a Typo Error
    corrupted_address = address[:-1] + ("x" if address[-1] != "x" else "y")
    print(f"\nTesting Corrupted Address: {corrupted_address}")
    try:
        b58check_decode(corrupted_address)
    except ValueError as e:
        print(f"Caught expected error: {e}")

Encoded Address: 1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAs
Validation Result: Success!
Version Byte: 0x00
Payload: f54a5851e9372b87810a8e60cdd2e7cfd80b6e31

Testing Corrupted Address: 1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAx
Caught expected error: Invalid Base58Check checksum! Data is corrupt or mistyped.
